<a href="https://colab.research.google.com/github/scarlettyu2023/AI_agent_workshop/blob/main/Topic6VLM/Task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install -U "transformers>=4.39" "accelerate>=0.26" bitsandbytes langgraph gradio pillow

In [ ]:
!pip uninstall -y pillow
!pip install pillow==10.3.0

In [ ]:
!git config --global credential.helper store
!hf auth login

In [ ]:
import torch
from typing import TypedDict, List, Dict, Any, Optional

import gradio as gr
from PIL import Image

from langgraph.graph import StateGraph, START, END

from transformers import AutoTokenizer, CLIPImageProcessor, LlavaProcessor, LlavaForConditionalGeneration

MODEL_ID = "bczhou/tiny-llava-v1-hf"

def load_tiny_llava():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)

    # Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

    # Image processor (CLIP)
    image_processor = CLIPImageProcessor.from_pretrained(MODEL_ID)

    # Build LlavaProcessor manually
    processor = LlavaProcessor(tokenizer=tokenizer, image_processor=image_processor)

    # IMPORTANT: ensure patch_size is set (some tiny checkpoints miss it)
    # Most LLaVA CLIP backbones use patch_size=14.
    if getattr(processor, "patch_size", None) is None:
        processor.patch_size = 14

    model = LlavaForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map="auto" if device == "cuda" else None,
    )
    if device != "cuda":
        model = model.to(device)


    model.eval()
    print("Model loaded:", MODEL_ID)
    return processor, model, device

processor, model, device = load_tiny_llava()

# ============================================================
# LangGraph State Definition
# ============================================================

class AgentState(TypedDict):
    messages: List[Dict[str, Any]]   # conversation history
    user_input: str                  # latest user message
    assistant_output: str            # latest model response
    image: Optional[Image.Image]     # uploaded image (persistent)


SYSTEM_PROMPT = (
    "You are a helpful vision-language assistant. "
    "Answer questions using the image and the conversation context. "
    "If something is not visible in the image, say you are not sure."
)

# ============================================================
# Prompt Builder
# ============================================================

def build_prompt_from_history(messages: List[Dict[str, Any]], user_text: str) -> str:
    """
    Build a compact text prompt from multi-turn history.
    LLaVA expects text + image input.
    """
    lines = []
    for m in messages:
        role = m["role"]
        content = m["content"]

        if role == "system":
            lines.append(f"[SYSTEM] {content}")
        elif role == "user":
            lines.append(f"User: {content}")
        elif role == "assistant":
            lines.append(f"Assistant: {content}")

    lines.append(f"User: {user_text}")
    lines.append("Assistant:")
    return "\n".join(lines)


# ============================================================
# LLaVA Inference
# ============================================================

def llava_generate(image: Image.Image, history: List[Dict[str, Any]], user_text: str, max_new_tokens: int = 128) -> str:
    """
    TinyLLaVA does not provide a chat template, so we manually format the prompt.
    IMPORTANT: include the <image> placeholder token so image tokens match features.
    """
    # Build a compact conversation prompt
    lines = []
    for m in history:
        role = m["role"]
        content = m["content"]

        if role == "system":
            # Keep system instruction minimal
            lines.append(f"SYSTEM: {content}")
        elif role == "user":
            lines.append(f"USER: {content}")
        elif role == "assistant":
            lines.append(f"ASSISTANT: {content}")

    # Current user turn MUST include <image>
    lines.append(f"USER: <image>\n{user_text}")
    lines.append("ASSISTANT:")

    prompt = "\n".join(lines)

    inputs = processor(text=prompt, images=image, return_tensors="pt")
    for k in inputs:
        inputs[k] = inputs[k].to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )

    decoded = processor.decode(output_ids[0], skip_special_tokens=True)

    # Extract only the assistant part (best-effort)
    if "ASSISTANT:" in decoded:
        decoded = decoded.split("ASSISTANT:")[-1].strip()

    return decoded.strip()


# ============================================================
# LangGraph Construction
# ============================================================

def create_graph():

    def call_vlm(state: AgentState) -> dict:
        user_text = (state["user_input"] or "").strip()
        if user_text == "":
            user_text = "Please describe the image."

        image = state.get("image", None)
        if image is None:
            return {"assistant_output": "Please upload an image first."}

        history = state["messages"]
        if not history or history[0].get("role") != "system":
            history = [{"role": "system", "content": SYSTEM_PROMPT}] + history

        answer = llava_generate(image, history, user_text, max_new_tokens=128)

        new_history = history + [
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": answer},
        ]

        return {
            "assistant_output": answer,
            "messages": new_history,
        }

    graph_builder = StateGraph(AgentState)
    graph_builder.add_node("call_vlm", call_vlm)
    graph_builder.add_edge(START, "call_vlm")
    graph_builder.add_edge("call_vlm", END)

    return graph_builder.compile()


graph = create_graph()

# ============================================================
# Initial State
# ============================================================

def init_state() -> AgentState:
    return {
        "messages": [{"role": "system", "content": SYSTEM_PROMPT}],
        "user_input": "",
        "assistant_output": "",
        "image": None,
    }

# ============================================================
# Gradio Chat Interface
# ============================================================

def on_chat(user_text, image, state, chat_history):
    if state is None or state == {}:
        state = init_state()

    # First turn requires image
    if state["image"] is None:
        if image is None:
            chat_history = chat_history or []
            chat_history.append(("System", "Please upload an image first."))
            return chat_history, state

        pil = image.convert("RGB")

        # Resize large images for speed
        max_side = 1024
        w, h = pil.size
        if max(w, h) > max_side:
            scale = max_side / float(max(w, h))
            pil = pil.resize((int(w * scale), int(h * scale)))

        state["image"] = pil

    state["user_input"] = user_text
    out = graph.invoke(state)
    state.update(out)

    chat_history = chat_history or []
    chat_history.append((user_text, state["assistant_output"]))
    return chat_history, state


def on_clear():
    return [], init_state()


with gr.Blocks() as demo:
    gr.Markdown("# Exercise 1 — Vision-Language LangGraph Chat Agent")

    with gr.Row():
        image_input = gr.Image(type="pil", label="Upload Image (required once)")
        state_store = gr.State(init_state())

    chatbot = gr.Chatbot(height=420)
    text_input = gr.Textbox(
        label="Your message",
        placeholder="Ask something about the image...",
        lines=2
    )

    with gr.Row():
        send_btn = gr.Button("Send")
        clear_btn = gr.Button("Clear")

    send_btn.click(
        on_chat,
        inputs=[text_input, image_input, state_store, chatbot],
        outputs=[chatbot, state_store],
    )

    text_input.submit(
        on_chat,
        inputs=[text_input, image_input, state_store, chatbot],
        outputs=[chatbot, state_store],
    )

    clear_btn.click(on_clear, inputs=None, outputs=[chatbot, state_store])

demo.queue().launch(share=True, debug=True, show_error=True)

# Exercise 1 — Vision-Language LangGraph Chat Agent

## Goal
Build a **multi-turn chat agent** that can discuss an **uploaded image**. The agent must:
- Accept an image upload
- Support **multi-turn** conversation about the image
- Use **LangGraph best practices** for structure and context/state management
- Mitigate slow performance by **reducing image resolution** when needed

---

## What I Built
I implemented a vision-language chat system with:
- **Gradio UI** for image upload + chat
- **HuggingFace Vision-Language Model** (e.g., TinyLLaVA or LLaVA) for image-conditioned responses
- **LangGraph** to manage state, conversation memory, and turn-by-turn execution

Users upload an image once, then ask follow-up questions like:
- “What objects do you see?”
- “What color is the main item?”
- “What is the person doing?”
The agent keeps context across turns and answers consistently.

---

## LangGraph Design (Good Style)
### State Definition
The agent uses a typed state object that explicitly captures all information needed across turns:

- `image`: the uploaded image (persisted across turns)
- `messages`: conversation history (user/assistant turns)
- `user_input`: the newest user message
- `assistant_output`: the newest model response

This makes data flow explicit and prevents ad-hoc global variables.

### Graph Structure
Each user turn runs the graph once:
